In [ ]:
# EPL Match Predictor - Training and Prediction Demo
# This notebook demonstrates the complete workflow: feature engineering, training, and prediction

import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path().resolve().parent))

from src import features, model, predict
import pandas as pd
import numpy as np



In [ ]:
# Load matches data
matches_df = features.load_matches("../data/raw/matches.csv")
print(f"Loaded {len(matches_df)} matches")
matches_df.head()



In [ ]:
# Engineer features
print("Engineering features...")
processed_df, X, y = features.prepare_features(matches_df, window=3)

print(f"Processed {len(processed_df)} matches")
print(f"Features: {X.columns.tolist()}")
print(f"Target distribution: {y.value_counts().to_dict()}")



In [ ]:
# Split into train and test sets
cutoff_date = '2022-01-01'
train_df, test_df = model.split_by_date(processed_df, cutoff_date)

print(f"Training set: {len(train_df)} matches")
print(f"Test set: {len(test_df)} matches")

# Get features for train/test
X_train = X.loc[train_df.index]
y_train = y.loc[train_df.index]
X_test = X.loc[test_df.index]
y_test = y.loc[test_df.index]



In [ ]:
# Train model
print("Training Random Forest model...")
trained_model, metrics = model.train_and_evaluate(
    X_train, y_train, X_test, y_test,
    model_type='random_forest',
    save_path="../models/model.joblib"
)

print("\n=== Model Evaluation ===")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")



In [ ]:
# Display confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = np.array(metrics['confusion_matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Loss/Draw', 'Win'], 
            yticklabels=['Loss/Draw', 'Win'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()



In [ ]:
# Example predictions
print("=== Example Predictions ===\n")

# Load the saved model
loaded_model = model.load_model("../models/model.joblib")

# Predict a few matches
test_matches = [
    ("Arsenal", "Liverpool"),
    ("Manchester City", "Tottenham"),
    ("Chelsea", "Manchester Utd"),
]

for home, away in test_matches:
    result = predict.predict_match(loaded_model, home, away, processed_df)
    print(f"{result['home_team']} vs {result['away_team']}")
    print(f"  Predicted: {result['predicted']}")
    print(f"  Home Win: {result['home_win_prob']:.2%}, Draw: {result['draw_prob']:.2%}, Away Win: {result['away_win_prob']:.2%}")
    print()

